> **Status: PREPARED — NOT YET EXECUTED**  
> Paths updated 2026-06-26 after repository reorganisation.  
> Run A1_01 first to produce `04_graph_dataset/nodes.csv` and `edges.csv`.

# MaCAD S.3 — Assignment 1
## Graph Visualisation and Export: Double-L Residential Building

**Run Notebook 1 first** (`A1_01_DoubleL_Geometry_to_Graph.ipynb`) to produce `nodes.csv` and `edges.csv`.

**Course workflow:** S02 — Geometry to Topology  
**Primary reference:** `S02-01 Primal vs Dual.ipynb` (graph visualisation), `S03-07` (degree preview).

This notebook reads those CSV files and generates:

| Figure | Filename | Colours by |
|--------|----------|------------|
| 2D plan view — space type | `02_graph_by_type.png` | space_type |
| 2D plan view — floor level | `03_graph_by_floor.png` | floor_id |
| Apartment cluster map | `04_graph_by_apartment.png` | apartment_id |
| Degree distribution | `05_degree_distribution.png` | — |
| Assignment report summary | `assignment1_report_summary.txt` | — |

### Assignment 1 Requirement Checklist (A1-02)

| # | Requirement | Status |
|---|-------------|--------|
| 1 | Load A1 graph from 04_graph_dataset/ | \[ \] Run to confirm |
| 2 | Graph coloured by space_type | \[ \] Run to confirm |
| 3 | Graph coloured by floor_id | \[ \] Run to confirm |
| 4 | Graph coloured by apartment_id | \[ \] Run to confirm |
| 5 | Degree distribution plot | \[ \] Run to confirm |
| 6 | Corridor betweenness analysis printed | \[ \] Run to confirm |
| 7 | Apartment connectivity check printed | \[ \] Run to confirm |
| 8 | All PNG figures saved to 05_visuals/ | \[ \] Run to confirm |
| 9 | Report summary text file saved | \[ \] Run to confirm |

---
## 1. Setup

In [1]:
import os
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize

matplotlib.rcParams["figure.dpi"] = 120
matplotlib.rcParams["font.family"] = "sans-serif"

# ── Path configuration ─────────────────────────────────────────────────────
PROJECT_ROOT = "D:/GitHub/GML_Edu"

# Assignment 1 root (moved to assignments/ in 2026-06-26 reorganisation)
ASSIGN_ROOT  = os.path.join(PROJECT_ROOT, "assignments", "assignment_01_graph_generation")
OUTPUTS_DIR  = os.path.join(ASSIGN_ROOT, "04_graph_dataset")
VISUALS_DIR  = os.path.join(ASSIGN_ROOT, "05_visuals")

os.makedirs(VISUALS_DIR, exist_ok=True)

NODES_CSV = os.path.join(OUTPUTS_DIR, "nodes.csv")
EDGES_CSV = os.path.join(OUTPUTS_DIR, "edges.csv")

for label, path in [("nodes.csv", NODES_CSV), ("edges.csv", EDGES_CSV)]:
    status = "OK" if os.path.exists(path) else "MISSING — run A1_01 first"
    print(f"{label}: [{status}]")

if not os.path.exists(NODES_CSV) or not os.path.exists(EDGES_CSV):
    raise FileNotFoundError(
        "Graph dataset not found.\n"
        f"Expected: {OUTPUTS_DIR}\n"
        "Run A1_01_DoubleL_Geometry_to_Graph.ipynb first."
    )

nodes.csv: [MISSING — run A1_01 first]
edges.csv: [MISSING — run A1_01 first]


FileNotFoundError: Graph dataset not found.
Expected: D:/GitHub/GML_Edu\assignments\assignment_01_graph_generation\04_graph_dataset
Run A1_01_DoubleL_Geometry_to_Graph.ipynb first.

---
## 2. Load Graph Data

In [ ]:
nodes = pd.read_csv(NODES_CSV)
edges = pd.read_csv(EDGES_CSV)

print(f"Nodes loaded: {len(nodes)}")
print(f"Edges loaded: {len(edges)}")
print(f"Columns (nodes): {list(nodes.columns)}")
print(f"Columns (edges): {list(edges.columns)}")
print()
print("Nodes by space_type:")
print(nodes["space_type"].value_counts().to_string())

In [ ]:
# Build NetworkX graph
G = nx.Graph()

for _, row in nodes.iterrows():
    G.add_node(int(row["node_id"]), **row.to_dict())

for _, row in edges.iterrows():
    G.add_edge(int(row["src_id"]), int(row["dst_id"]))

print(f"NetworkX graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Connected components  : {nx.number_connected_components(G)}")
print(f"Isolated nodes        : {len(list(nx.isolates(G)))}")
avg_deg = sum(d for _, d in G.degree()) / G.number_of_nodes() if G.number_of_nodes() > 0 else 0
print(f"Average degree        : {avg_deg:.2f}")
print(f"Max degree            : {max(d for _, d in G.degree()) if G.number_of_nodes() > 0 else 0}")
print(f"Graph density         : {nx.density(G):.4f}")

---
## 3. Colour Palettes and Layouts

In [ ]:
# Space-type colour palette (matches Notebook 1)
TYPE_COLORS = {
    "Hall"     : "#FF9800",
    "Bath"     : "#CE93D8",
    "Kitchen"  : "#EF9A9A",
    "Living"   : "#4FC3F7",
    "Bedroom"  : "#B3E5FC",
    "Bedroom1" : "#81D4FA",
    "Bedroom2" : "#4DD0E1",
    "Corridor" : "#81C784",
    "Core"     : "#E57373",
    "unknown"  : "#EEEEEE",
}

# Node size by role
ROLE_SIZES = {"room": 60, "corridor": 160, "core": 120}

# Use XY centroid as 2D layout (plan view — metres)
pos_xy = {}
for n, attrs in G.nodes(data=True):
    x = float(attrs.get("X", 0))
    y = float(attrs.get("Y", 0))
    pos_xy[n] = (x, y)

node_colors = [TYPE_COLORS.get(G.nodes[n].get("space_type", "unknown"), "#EEEEEE")
               for n in G.nodes()]
node_sizes  = [ROLE_SIZES.get(G.nodes[n].get("node_role", "room"), 60)
               for n in G.nodes()]

print("Colour palette and positions ready.")

---
## 4. Figure 1 — Graph Coloured by Space Type

Each node's colour indicates its architectural programme:
Hall (orange) is the apartment entry point and should connect to bath, kitchen, living, and corridor.
Corridors (green) appear as high-degree hubs.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

nx.draw_networkx_edges(G, pos_xy, ax=ax, edge_color="#BBBBBB", width=0.5, alpha=0.6)
nx.draw_networkx_nodes(G, pos_xy, ax=ax,
                       node_color=node_colors,
                       node_size=node_sizes,
                       edgecolors="#333333", linewidths=0.4)

present_types = nodes["space_type"].unique()
legend_patches = [mpatches.Patch(color=c, label=k)
                  for k, c in TYPE_COLORS.items() if k in present_types]
ax.legend(handles=legend_patches, loc="upper right", fontsize=8, framealpha=0.9,
          title="Space type")

ax.set_title("Double-L Residential — Spatial Graph by Space Type", fontsize=13, pad=12)
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.set_aspect("equal")
plt.tight_layout()

out_path = os.path.join(VISUALS_DIR, "02_graph_by_type.png")
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

---
## 5. Figure 2 — Graph Coloured by Floor

Floor-level colouring reveals the vertical repetition of the Double-L plan.
If cross-floor edges appear (corridors connecting adjacent floors via cores), they will
span between colour bands.

In [ ]:
floors     = sorted(nodes["floor_id"].dropna().unique())
floor_cmap = plt.cm.viridis
floor_norm = Normalize(vmin=0, vmax=max(floors) if len(floors) > 1 else 1)

def floor_color(n):
    fl = G.nodes[n].get("floor_id")
    if fl is None:
        return "#EEEEEE"
    return floor_cmap(floor_norm(float(fl)))

node_colors_f = [floor_color(n) for n in G.nodes()]

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_edges(G, pos_xy, ax=ax, edge_color="#CCCCCC", width=0.5, alpha=0.6)
nx.draw_networkx_nodes(G, pos_xy, ax=ax,
                       node_color=node_colors_f,
                       node_size=node_sizes,
                       edgecolors="#333333", linewidths=0.3)

sm = plt.cm.ScalarMappable(cmap=floor_cmap, norm=floor_norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Floor ID", fontsize=9)
cbar.set_ticks(floors)

ax.set_title("Double-L Residential — Spatial Graph by Floor Level", fontsize=13, pad=12)
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.set_aspect("equal")
plt.tight_layout()

out_path = os.path.join(VISUALS_DIR, "03_graph_by_floor.png")
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

---
## 6. Figure 3 — Apartment Cluster Map

Each apartment receives a distinct hue. Corridor and core nodes remain grey.
This reveals how the 23-apartment-per-floor arrangement spatially clusters and how
each cluster connects to the shared circulation spine.

In [ ]:
apt_ids = sorted(
    [a for a in nodes["apartment_id"].dropna().unique() if str(a).strip() != ""]
)
n_apts   = len(apt_ids)
apt_cmap  = cm.get_cmap("tab20", max(n_apts, 1))
apt_color = {apt: matplotlib.colors.to_hex(apt_cmap(i))
             for i, apt in enumerate(apt_ids)}

def apt_node_color(n):
    apt = G.nodes[n].get("apartment_id", "")
    if not apt or str(apt).strip() == "":
        return "#CCCCCC"
    return apt_color.get(apt, "#EEEEEE")

node_colors_a = [apt_node_color(n) for n in G.nodes()]

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_edges(G, pos_xy, ax=ax, edge_color="#CCCCCC", width=0.4, alpha=0.5)
nx.draw_networkx_nodes(G, pos_xy, ax=ax,
                       node_color=node_colors_a,
                       node_size=node_sizes,
                       edgecolors="#555555", linewidths=0.3)

sample_apts = apt_ids[:20]
legend_patches = [mpatches.Patch(color=apt_color[a], label=str(a)) for a in sample_apts]
if n_apts > 20:
    legend_patches.append(mpatches.Patch(color="none", label=f"\u2026 +{n_apts - 20} more"))
ax.legend(handles=legend_patches, loc="upper right", fontsize=6,
          framealpha=0.85, ncol=2, title="Apartment ID")

ax.set_title("Double-L Residential — Apartment Clustering", fontsize=13, pad=12)
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.set_aspect("equal")
plt.tight_layout()

out_path = os.path.join(VISUALS_DIR, "04_graph_by_apartment.png")
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

---
## 7. Figure 4 — Degree Distribution

Degree = number of directly adjacent rooms sharing a wall boundary.

Expected pattern for a residential building with shared-face adjacency:
- Hall nodes: moderate degree (connects to Bath, Kitchen, Living, and corridor)
- Bath / Bedroom nodes: low degree (fewer shared walls)
- Corridor nodes: high degree (shared wall with multiple apartment Halls)

In [ ]:
degree_by_type = {}
for n, deg in G.degree():
    st = G.nodes[n].get("space_type", "unknown")
    degree_by_type.setdefault(st, []).append(deg)

all_degrees = [d for _, d in G.degree()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

max_deg = max(all_degrees) if all_degrees else 1
axes[0].hist(all_degrees, bins=range(0, max_deg + 2), color="#78909C",
             edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Degree (shared-face neighbours)", fontsize=10)
axes[0].set_ylabel("Count", fontsize=10)
axes[0].set_title("Degree Distribution — All Nodes", fontsize=11)
mean_deg = np.mean(all_degrees) if all_degrees else 0
axes[0].axvline(mean_deg, color="red", linestyle="--", lw=1.5,
                label=f"Mean = {mean_deg:.1f}")
axes[0].legend(fontsize=9)

type_means   = {st: np.mean(degs) for st, degs in degree_by_type.items() if degs}
sorted_types = sorted(type_means, key=lambda x: type_means[x], reverse=True)
bar_colors   = [TYPE_COLORS.get(st, "#EEEEEE") for st in sorted_types]
bars = axes[1].bar(sorted_types, [type_means[st] for st in sorted_types],
                   color=bar_colors, edgecolor="#333333", linewidth=0.5)
axes[1].set_ylabel("Mean Degree", fontsize=10)
axes[1].set_title("Mean Degree by Space Type", fontsize=11)
axes[1].tick_params(axis="x", rotation=35, labelsize=9)
for bar, st in zip(bars, sorted_types):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                 f"{type_means[st]:.1f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("Spatial Connectivity Analysis — Double-L Residential", fontsize=12, y=1.01)
plt.tight_layout()

out_path = os.path.join(VISUALS_DIR, "05_degree_distribution.png")
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

---
## 8. Spatial Analysis

Use these cells to generate insights for your written submission.

In [ ]:
# Top-degree nodes — which spaces are topologically most connected?
top_n = 10
top_nodes = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:top_n]
print(f"Top {top_n} nodes by degree:")
print(f"{'node':>5}  {'deg':>4}  {'type':12}  {'role':10}  {'apt':20}  {'floor'}")
print("-" * 70)
for n, deg in top_nodes:
    attrs = G.nodes[n]
    print(f"{n:>5}  {deg:>4}  "
          f"{str(attrs.get('space_type','?')):12}  "
          f"{str(attrs.get('node_role','?')):10}  "
          f"{str(attrs.get('apartment_id','')):20}  "
          f"{attrs.get('floor_id','?')}")

In [ ]:
# Apartment internal connectivity — are all rooms in an apartment reachable from one another?
apt_col = "apartment_id"
apt_nodes_df = nodes[(nodes[apt_col].notna()) & (nodes[apt_col].astype(str).str.strip() != "")]

print("Apartment internal connectivity:")
fragmented = []
for apt_id, group in apt_nodes_df.groupby(apt_col):
    ids = set(group["node_id"].astype(int))
    sub = G.subgraph(ids)
    comps = nx.number_connected_components(sub)
    iso   = len(list(nx.isolates(sub)))
    if comps != 1 or iso > 0:
        status = f"FRAGMENTED ({comps} components, {iso} isolated)"
        fragmented.append(apt_id)
    else:
        status = "connected"
    print(f"  {str(apt_id):22s}  rooms={len(ids):3d}  [{status}]")

print()
if fragmented:
    print(f"Fragmented apartments: {len(fragmented)}")
    print("  This is expected when rooms share no exact face — see Notebook 1 blocker notes.")
else:
    print("All apartments are internally connected.")

In [ ]:
# Corridor betweenness centrality — which corridors are most critical?
corr_nodes = [n for n in G.nodes() if G.nodes[n].get("node_role") == "corridor"]

if corr_nodes and G.number_of_edges() > 0:
    between = nx.betweenness_centrality(G, normalized=True)
    print("Corridor betweenness centrality (0=peripheral, 1=most central):")
    for n in sorted(corr_nodes, key=lambda x: between[x], reverse=True):
        attrs = G.nodes[n]
        name  = attrs.get("node_name", str(n))
        fl    = attrs.get("floor_id", "?")
        print(f"  {str(name):28s}  floor={fl}  betweenness={between[n]:.4f}")
elif not corr_nodes:
    print("No corridor nodes found — check node_role attribute in A1_01.")
else:
    print("Graph has no edges — run A1_01 and check adjacency detection.")

---
## 9. Assignment 1 Report Summary

In [ ]:
n_rooms  = len(nodes[nodes["node_role"] == "room"]) if "node_role" in nodes.columns else len(nodes)
n_corr   = len(nodes[nodes["node_role"] == "corridor"]) if "node_role" in nodes.columns else 0
n_core   = len(nodes[nodes["node_role"] == "core"]) if "node_role" in nodes.columns else 0
n_floors = nodes["floor_id"].nunique() if "floor_id" in nodes.columns else 0
n_apts_t = len(apt_ids)

report = f"""
====================================================
  MaCAD S.3 — Assignment 1: Graph Report
  Double-L Residential Building — TB_01
====================================================
BUILDING MODEL
  Type          : Double-L residential with pilotis
  Floors        : {n_floors}
  Source        : resident_gen TB_01 exports

GRAPH METHODOLOGY
  Method        : Dual graph — shared-face adjacency
  API (course)  : Graph.ByTopology(model, direct=True)  [S02-01]
  Edge semantics: Two cells sharing an exact wall boundary

  Door apertures: UNAVAILABLE (0 door objects in TB_01_doors.obj)
  Workaround    : Shared-face adjacency as topological proxy

NODES
  Total nodes   : {G.number_of_nodes()}
    Room nodes  : {n_rooms}
    Corridor    : {n_corr}
    Core        : {n_core}
  Apartments    : {n_apts_t} unique IDs across all floors

EDGES
  Total edges   : {G.number_of_edges()}
  Average degree: {np.mean([d for _, d in G.degree()]):.2f}
  Max degree    : {max((d for _, d in G.degree()), default=0)}

CONNECTIVITY
  Components    : {nx.number_connected_components(G)}
  Isolated nodes: {len(list(nx.isolates(G)))}

NODE ATTRIBUTES
  space_type    : Hall, Bath, Kitchen, Living, Bedroom,
                  Bedroom1, Bedroom2, Corridor, Core
  zone_type     : circulation | private | service
  node_role     : room | corridor | core
  apartment_id  : per-room apartment identifier
  floor_id      : 0 – {n_floors - 1 if n_floors else '?'}
  area, volume  : from TB_01_metadata.csv
  X, Y, Z       : graph vertex centroid position (metres)

KNOWN LIMITATIONS
  1. TB_01_doors.obj is empty (0 objects). Door-access graph
     via apertures (S02-03 directApertures) is unavailable.
  2. Rooms separated by a wall of non-zero thickness will have
     no graph edge — only exact face coincidence produces edges.
  3. Corridor connector / stair geometry not in TB_01 exports.
     Vertical inter-floor links are therefore not encoded.

OUTPUT FILES
  04_graph_dataset/nodes.csv
  04_graph_dataset/edges.csv
  05_visuals/01_geometry_and_graph.png  (from A1_01)
  05_visuals/02_graph_by_type.png
  05_visuals/03_graph_by_floor.png
  05_visuals/04_graph_by_apartment.png
  05_visuals/05_degree_distribution.png
====================================================
"""

print(report)
report_path = os.path.join(VISUALS_DIR, "assignment1_report_summary.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)
print(f"Report saved: {report_path}")

---
## Node / Edge Mapping Explanation

Include this interpretation in your submission:

| Spatial element | Graph element | Attribute encoded |
|-----------------|--------------|------------------|
| Room (Hall, Bath, Kitchen, Living, Bedroom) | Node | space_type, zone_type, apartment_id, floor_id, area |
| Corridor segment | Node (high-degree hub) | node_role = corridor, floor_id |
| Core (stair / lift) | Node | node_role = core |
| Shared wall face between two rooms | Edge | edge_type = adjacency |

**Why this graph?** The shared-face adjacency graph encodes the topological skeleton of
the building — which spaces touch which. With 23 apartments per floor and a central
corridor spine, the graph reveals the Double-L's characteristic "double-loaded corridor"
structure: each corridor node connects to the Halls of all apartments on that floor.

---
## Final Submission Checklist

- [ ] All PNG figures saved to `05_visuals/`
- [ ] `assignment1_report_summary.txt` saved
- [ ] Degree distribution plot shows non-trivial spread
- [ ] Betweenness centrality printed for corridors
- [ ] Apartment connectivity check shows results
- [ ] Copy `04_graph_dataset/nodes.csv` + `edges.csv` to `../assignment_02_graph_analysis/01_input_graph/` before running Assignment 2